# LLM과 Transformer 아키텍처

## 소속 및 성명

반: 4반

번호: U110

성명: 김근홍

## 다음에 제시한 주제에 대하여 300~500자 이내로 본인의 생각을 정리하여 작성하세요

AI 시대에 접어들며 소프트웨어 개발은 많은 변화를 겪게 되었습니다. 소프트웨어 개발에 있어 AI를 사용하는 것에 대한 개인적인 입장을 서술하세요

In [11]:
text="""
소프트웨어 개발에서의 AI를 사용하는 것은 장단점이 분명히 존재한다고 생각합니다.

장점
1. 요구사항이나 기획서를 아이디어를 빠르게 시험해볼 수 있습니다. 이를 통해 비용이나 시간을 아낄 수 있습니다.
2. 빠르게 동시에 작업을 할 수 있습니다. 하나의 기능만을 만드는 것이 아닌 동시에 여러 작업을 할 수 있어, 개발 시간 단축으로 이어집니다.
3. 테스트가 간편해집니다. 유닛 혹은 통합 테스트 코드를 자연어 명령으로 쉽게 테스트 진행이 가능해져 코드의 문제점을 빠르게 파악할 수 있습니다.

단점
1. 코드가 대량으로 생성되는 만큼, 개발자가 그 코드를 검토하는 시간이 배로 걸리게 됩니다.
2. 빠르게 작성되고 버려지는 코드가 늘어납니다. 사용하지 않는 코드가 문서 어딘가에 남아있습니다.
3. 개발자가 도메인에 대해 이해하지 못할 경우 환각이 일어났는지에 대한 판별이 불가능합니다.
"""

In [12]:
print(len(text))

445


## 아래의 실습 코드를 실행해보세요

Bag of Words

In [13]:
from sklearn.feature_extraction.text import CountVectorizer

docs = ["나는 사과를 좋아한다", "나는 바나나를 좋아한다"]
vectorizer = CountVectorizer()
vectors = vectorizer.fit_transform(docs)

print(vectorizer.get_feature_names_out())
print(vectors.toarray())


['나는' '바나나를' '사과를' '좋아한다']
[[1 0 1 1]
 [1 1 0 1]]


Word2Vec

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 90.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [gensim]2m2/3 [gensim]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [28]:
from gensim.models import Word2Vec

# 1. 말뭉치(Corpus) 정의: 토큰화된 문장들의 리스트
corpus = [
    ["인공지능", "딥러닝", "머신러닝", "학습", "데이터"],
    ["자연어", "처리", "트랜스포머", "임베딩", "토크나이저"],
    ["컴퓨터", "비전", "이미지", "합성곱", "신경망"],
    ["딥러닝", "기반", "자연어", "처리", "모델", "학습"],
    ["왕", "남자", "통치자", "궁궐"],
    ["여왕", "여자", "통치자", "궁궐"],
    ["왕자", "남자", "어린", "궁궐"],
    ["공주", "여자", "어린", "궁궐"],
]

# 2. Word2Vec 모델 생성 및 학습
model = Word2Vec(
    sentences=corpus,
    vector_size=1300,  # 임베딩 벡터 차원 수
    window=9,         # 문맥 윈도우 크기 (앞뒤 3개 단어 참조)
    min_count=1,      # 최소 빈도수 (1회 이상 등장한 단어 모두 포함)
    workers=4,        # 학습에 사용할 스레드 수
    sg=1,             # 0: CBOW (주변 단어로 중심 단어 예측), 1: Skip-Gram (중심 단어로 주변 단어 예측)
    epochs=5000        # 학습 반복 횟수
)

# 3. 특정 단어의 임베딩 벡터 확인
target_word = "딥러닝"
vector = model.wv[target_word]
print(f"'{target_word}'의 임베딩 벡터 형태(Shape): {vector.shape}")
print(f"'{target_word}' 벡터 샘플 (앞 5개 차원): {vector[:5]}\n")

# 4. 코사인 유사도 기반 가장 유사한 단어 검색
similar_words = model.wv.most_similar("자연어", topn=3)
print("▶ '자연어'와 가장 유사한 단어 Top 3:")
for word, score in similar_words:
    print(f"  - {word}: {score:.4f}")
print()

# 5. 두 단어 간 유사도 직접 계산
similarity = model.wv.similarity("왕", "여왕")
print(f"▶ '왕'과 '여왕'의 코사인 유사도: {similarity:.4f}\n")

# 6. 단어 벡터 유추 연산 (Analogy: 왕 - 남자 + 여자 = ?)
analogy_result = model.wv.most_similar(positive=["왕", "여자"], negative=["남자"], topn=10)
print("▶ '왕' - '남자' + '여자' 연산 결과:")
for word, score in analogy_result:
    print(f"  - {word}: {score:.4f}")

# 7. 모델 저장 및 로드
model.save("word2vec_model.bin")
# loaded_model = Word2Vec.load("word2vec_model.bin")

'딥러닝'의 임베딩 벡터 형태(Shape): (1300,)
'딥러닝' 벡터 샘플 (앞 5개 차원): [-0.0207126   0.15888895 -0.04619911  0.02628554 -0.00942219]

▶ '자연어'와 가장 유사한 단어 Top 3:
  - 처리: 0.9975
  - 기반: 0.9539
  - 모델: 0.9522

▶ '왕'과 '여왕'의 코사인 유사도: 0.9966

▶ '왕' - '남자' + '여자' 연산 결과:
  - 여왕: 0.9948
  - 왕자: 0.9939
  - 통치자: 0.9935
  - 공주: 0.9905
  - 어린: 0.9904
  - 궁궐: 0.9869
  - 이미지: 0.4990
  - 신경망: 0.4977
  - 컴퓨터: 0.4928
  - 합성곱: 0.4917


Tokenizer, Vocabulary, Embedding

별도 파일

Attention

별도 파일

Corpus

In [ ]:
# 개념적 예시: 간단한 말뭉치 정제 파이프라인
raw_docs = ["이것은 좋은 문장입니다.", "광고 광고 광고 클릭하세요!!!", ""]

def clean(doc):
    if len(doc.strip()) < 5:
        return None
    if doc.count("광고") > 1:
        return None
    return doc.strip()

corpus = [clean(d) for d in raw_docs if clean(d)]
print(corpus)


실제 LLM 학습에서는 이러한 필터링이 **훨씬 대규모이고 정교한 규칙과 모델**로 수행됨